In [9]:
import pandas as pd
import plotly.express as px
from scipy.optimize import curve_fit
import numpy as np

# Load the CSV file into a DataFrame
raw_df = pd.read_csv('measurement.csv')
raw_df["voltage"] = raw_df["voltage"] / 1000  # convert mV to V

# Discard the last 6 clamped samples and reset index for clean plotting/fitting
df = raw_df.iloc[:-6].reset_index(drop=True).copy()
print(f"Dropped {len(raw_df) - len(df)} final samples; plotting {len(df)} points.")

# Fit an exponential decay curve
def exponential_decay(x, a, b, c):
    return a * np.exp(-b * x) + c

popt, _ = curve_fit(exponential_decay, df["DAC"], df["voltage"], p0=(3.3, 0.01, 0.5))

# Fit a polynomial curve
def polynomial(x, a, b, c):
    return a * x**2 + b * x + c

# fit a hyperbolic reciprocal function
def reciprocal(x, a, b, c):
    return a / (x + b) + c

popt_recip, _ = curve_fit(reciprocal, df["DAC"], df["voltage"])
popt_poly, _ = curve_fit(polynomial, df["DAC"], df["voltage"], p0=(0.01, -0.1, 3.3))

print("Exponential fit parameters:", popt)
print("Polynomial fit parameters:", popt_poly)
print("Reciprocal fit parameters:", popt_recip)

# Plot only the filtered data and fitted curves
df["fitted_voltage"] = exponential_decay(df["DAC"], *popt)
df["fitted_voltage_poly"] = polynomial(df["DAC"], *popt_poly)
df["fitted_voltage_recip"] = reciprocal(df["DAC"], *popt_recip)

fig = px.scatter(
    df,
    x="DAC",
    y="voltage",
    title="DAC Setpoint vs Voltage (First 5 Samples Removed)",
    labels={"DAC": "DAC Setpoint", "voltage": "Voltage (V)"}
 )
fig.add_scatter(x=df["DAC"], y=df["fitted_voltage"], mode='lines', name='Exponential Fit')
fig.add_scatter(x=df["DAC"], y=df["fitted_voltage_poly"], mode='lines', name='Polynomial Fit')
fig.add_scatter(x=df["DAC"], y=df["fitted_voltage_recip"], mode='lines', name='Reciprocal Fit')
fig.show()

df.head()

Dropped 6 final samples; plotting 249 points.
Exponential fit parameters: [2.61060958 0.01032385 0.8451743 ]
Polynomial fit parameters: [ 3.86520266e-05 -1.83380171e-02  3.26144337e+00]
Reciprocal fit parameters: [3.50435568e+02 9.78695166e+01 5.77986299e-03]


,DAC,voltage,fitted_voltage,fitted_voltage_poly,fitted_voltage_recip
0,255,1.005,1.032858,1.098597,0.998882
1,254,1.002,1.034806,1.097261,1.001705
2,253,1.006,1.036773,1.096003,1.004543
3,252,1.005,1.038762,1.094821,1.007398
4,251,1.012,1.040771,1.093717,1.010269


In [11]:
popt_recip

def convert_voltage_to_dac(voltage, a, b, c):
    return a / (voltage - c) - b

dac_value = convert_voltage_to_dac(2.0, *popt_recip)
print(f"Estimated DAC value for 2.0 V: {dac_value}")

Estimated DAC value for 2.0 V: 77.85610247858621
